In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [ ]:

# Step 1: Load the datasets
print("Step 1: Loading datasets...")
# Update these paths to match your file locations
food_category = pd.read_csv('FoodData_Central_csv_2024-10-31/food_category.csv')
food_nutrient = pd.read_csv('FoodData_Central_csv_2024-10-31/food_nutrient.csv')
food_branded = pd.read_csv('FoodData_Central_csv_2024-10-31/branded_food.csv')
food_attribute = pd.read_csv('FoodData_Central_csv_2024-10-31/food_attribute.csv')


In [ ]:

# Print basic information about the datasets
print(f"Food Category: {food_category.shape[0]} rows, {food_category.shape[1]} columns")
print(f"Food Nutrient: {food_nutrient.shape[0]} rows, {food_nutrient.shape[1]} columns")
print(f"Branded Food: {food_branded.shape[0]} rows, {food_branded.shape[1]} columns")
print(f"Food Attribute: {food_attribute.shape[0]} rows, {food_attribute.shape[1]} columns")


In [ ]:

# Print basic information about the datasets
print(f"Food Category: {food_category.shape[0]} rows, {food_category.shape[1]} columns")
print(f"Food Nutrient: {food_nutrient.shape[0]} rows, {food_nutrient.shape[1]} columns")
print(f"Branded Food: {food_branded.shape[0]} rows, {food_branded.shape[1]} columns")
print(f"Food Attribute: {food_attribute.shape[0]} rows, {food_attribute.shape[1]} columns")


In [ ]:

# Display the food categories to identify baby food category
print("\nFood categories related to baby foods:")
baby_categories = food_category[food_category['description'].str.contains('baby|infant|toddler', 
                                                                        case=False, na=False)]
print(baby_categories)

In [ ]:
# Step 2: Identify baby food products
print("\nStep 2: Identifying baby food products...")
baby_category_id = 300  # ID for "Baby Foods" as seen in the data

In [ ]:
# Get all foods in the baby food category
baby_foods_by_category = []
if 'food_category_id' in food_branded.columns:
    baby_foods_by_category = food_branded[food_branded['food_category_id'] == baby_category_id]
    print(f"Found {len(baby_foods_by_category)} baby foods by category ID")

# Also look for baby-related terms in ingredients/description/brand
baby_terms = ['baby', 'infant', 'toddler', 'gerber', 'beech-nut', 'earth\'s best']
baby_foods_by_keywords = pd.DataFrame()

for col in ['brand_name', 'ingredients', 'household_serving_fulltext']:
    if col in food_branded.columns:
        mask = food_branded[col].astype(str).str.contains('|'.join(baby_terms), case=False, na=False)
        baby_foods_by_keywords = pd.concat([baby_foods_by_keywords, food_branded[mask]])

print(f"Found {len(baby_foods_by_keywords)} baby foods by keywords")


In [ ]:

# Combine both sets and remove duplicates
baby_foods = pd.concat([baby_foods_by_category, baby_foods_by_keywords]).drop_duplicates(subset=['fdc_id'])
print(f"Total unique baby food products identified: {len(baby_foods)}")


In [ ]:
# Display a sample
print("\nSample of identified baby foods:")
sample_cols = ['fdc_id', 'brand_name', 'ingredients']
sample_cols = [col for col in sample_cols if col in baby_foods.columns]
print(baby_foods[sample_cols].head())


In [ ]:
# Step 3: Extract nutrient data for baby foods
print("\nStep 3: Extracting nutrient data for baby foods...")
# Get list of baby food IDs
baby_food_ids = baby_foods['fdc_id'].unique()
print(f"Getting nutrient data for {len(baby_food_ids)} baby food products")

In [ ]:


# Filter nutrient data for baby foods
baby_food_nutrients = food_nutrient[food_nutrient['fdc_id'].isin(baby_food_ids)].copy()
print(f"Found {len(baby_food_nutrients)} nutrient records for baby foods")

# Find unique nutrient IDs
unique_nutrients = baby_food_nutrients['nutrient_id'].unique()
print(f"Unique nutrient IDs in baby foods: {unique_nutrients}")

In [ ]:

# Potential sugar-related nutrient IDs (these may need adjustment based on your data)
sugar_nutrient_ids = [269, 539, 2000, 1235, 1082]  # These are IDs for various sugar measures

# Extract sugar content data
sugar_data = baby_food_nutrients[baby_food_nutrients['nutrient_id'].isin(sugar_nutrient_ids)]
print(f"Found {len(sugar_data)} sugar content records")

# Pivot the data to have one row per food with sugar content
sugar_pivot = sugar_data.pivot_table(
    index='fdc_id',
    columns='nutrient_id',
    values='amount',
    aggfunc='first'
).reset_index()

In [ ]:
# Rename columns for clarity
for col in sugar_pivot.columns:
    if col != 'fdc_id':
        sugar_pivot.rename(columns={col: f'nutrient_{col}'}, inplace=True)

# Merge with baby foods data
baby_foods_with_nutrients = pd.merge(
    baby_foods,
    sugar_pivot,
    on='fdc_id',
    how='left'
)

print(f"Final dataset has {len(baby_foods_with_nutrients)} rows and {baby_foods_with_nutrients.shape[1]} columns")


In [ ]:
# Step 4: Process ingredient information to create features
print("\nStep 4: Processing ingredients to create features...")

# Function to count occurrences of ingredients
def count_ingredient(ingredients, terms):
    if pd.isna(ingredients):
        return 0
    ingredients = str(ingredients).lower()
    return sum(1 for term in terms if term in ingredients)

# Function to find position of first occurrence
def first_position(ingredients, terms):
    if pd.isna(ingredients):
        return -1
    
    ingredients = str(ingredients).lower()
    ingredients_list = [i.strip() for i in ingredients.split(',')]
    
    for idx, ingredient in enumerate(ingredients_list):
        for term in terms:
            if term in ingredient:
                return idx
    return -1


In [ ]:

# Create features
df = baby_foods_with_nutrients.copy()

# Check for common sweeteners
sweeteners = [
    'sugar', 'corn syrup', 'fructose', 'glucose', 'dextrose', 'maltose', 
    'sucrose', 'honey', 'maple syrup', 'agave', 'molasses', 'juice concentrate'
]
df['sweetener_count'] = df['ingredients'].apply(lambda x: count_ingredient(x, sweeteners))
df['sweetener_position'] = df['ingredients'].apply(lambda x: first_position(x, sweeteners))


In [ ]:

# Check for fruits (often contain natural sugars)
fruits = [
    'apple', 'banana', 'pear', 'peach', 'mango', 'grape', 'berry', 'strawberry',
    'blueberry', 'raspberry', 'fruit', 'orange', 'plum', 'prune'
]
df['fruit_count'] = df['ingredients'].apply(lambda x: count_ingredient(x, fruits))
df['fruit_position'] = df['ingredients'].apply(lambda x: first_position(x, fruits))

# Ingredient count (complexity of the product)
df['ingredient_count'] = df['ingredients'].apply(
    lambda x: 0 if pd.isna(x) else len(str(x).split(','))
)

# Check for organic products
df['is_organic'] = df['ingredients'].apply(
    lambda x: 1 if pd.notna(x) and 'organic' in str(x).lower() else 0
)

In [ ]:
# Check for added preservatives
preservatives = [
    'preservative', 'citric acid', 'ascorbic acid', 'sodium benzoate', 
    'potassium sorbate', 'tocopherol', 'bht', 'bha'
]
df['preservative_count'] = df['ingredients'].apply(
    lambda x: count_ingredient(x, preservatives)
)

baby_foods_processed = df
print("Added ingredient-based features:")
print(f"  - sweetener_count: Count of sweetener ingredients")
print(f"  - sweetener_position: Position of first sweetener in ingredient list")
print(f"  - fruit_count: Count of fruit ingredients")
print(f"  - fruit_position: Position of first fruit in ingredient list")
print(f"  - ingredient_count: Total number of ingredients")
print(f"  - is_organic: Whether product is labeled organic")
print(f"  - preservative_count: Count of preservatives")


In [ ]:

# Step 4: Process ingredient information to create features
print("\nStep 4: Processing ingredients to create features...")

# Function to count occurrences of ingredients
def count_ingredient(ingredients, terms):
    if pd.isna(ingredients):
        return 0
    ingredients = str(ingredients).lower()
    return sum(1 for term in terms if term in ingredients)

# Function to find position of first occurrence
def first_position(ingredients, terms):
    if pd.isna(ingredients):
        return -1
    
    ingredients = str(ingredients).lower()
    ingredients_list = [i.strip() for i in ingredients.split(',')]
    
    for idx, ingredient in enumerate(ingredients_list):
        for term in terms:
            if term in ingredient:
                return idx
    return -1

# Create features
df = baby_foods_with_nutrients.copy()

# Check for common sweeteners
sweeteners = [
    'sugar', 'corn syrup', 'fructose', 'glucose', 'dextrose', 'maltose', 
    'sucrose', 'honey', 'maple syrup', 'agave', 'molasses', 'juice concentrate'
]
df['sweetener_count'] = df['ingredients'].apply(lambda x: count_ingredient(x, sweeteners))
df['sweetener_position'] = df['ingredients'].apply(lambda x: first_position(x, sweeteners))

# Check for fruits (often contain natural sugars)
fruits = [
    'apple', 'banana', 'pear', 'peach', 'mango', 'grape', 'berry', 'strawberry',
    'blueberry', 'raspberry', 'fruit', 'orange', 'plum', 'prune'
]
df['fruit_count'] = df['ingredients'].apply(lambda x: count_ingredient(x, fruits))
df['fruit_position'] = df['ingredients'].apply(lambda x: first_position(x, fruits))

# Ingredient count (complexity of the product)
df['ingredient_count'] = df['ingredients'].apply(
    lambda x: 0 if pd.isna(x) else len(str(x).split(','))
)

# Check for organic products
df['is_organic'] = df['ingredients'].apply(
    lambda x: 1 if pd.notna(x) and 'organic' in str(x).lower() else 0
)

# Check for added preservatives
preservatives = [
    'preservative', 'citric acid', 'ascorbic acid', 'sodium benzoate', 
    'potassium sorbate', 'tocopherol', 'bht', 'bha'
]
df['preservative_count'] = df['ingredients'].apply(
    lambda x: count_ingredient(x, preservatives)
)

baby_foods_processed = df
print("Added ingredient-based features:")
print(f"  - sweetener_count: Count of sweetener ingredients")
print(f"  - sweetener_position: Position of first sweetener in ingredient list")
print(f"  - fruit_count: Count of fruit ingredients")
print(f"  - fruit_position: Position of first fruit in ingredient list")
print(f"  - ingredient_count: Total number of ingredients")
print(f"  - is_organic: Whether product is labeled organic")
print(f"  - preservative_count: Count of preservatives")

# Find the best sugar content column (most non-null values)
sugar_cols = [col for col in baby_foods_processed.columns if 'nutrient_' in col]
print("\nPotential target variables (sugar content):")
for col in sugar_cols:
    print(f"{col}: {baby_foods_processed[col].count()} non-null values")

if sugar_cols:
    # Sort columns by non-null count
    sorted_cols = sorted([(col, baby_foods_processed[col].count()) 
                        for col in sugar_cols], key=lambda x: x[1], reverse=True)
    
    target_col = sorted_cols[0][0]
    print(f"\nSelected {target_col} as target variable with {sorted_cols[0][1]} non-null values")
    
    # Distribution of sugar content
    if baby_foods_processed[target_col].count() > 10:
        print("\nDistribution of sugar content in baby foods:")
        print(baby_foods_processed[target_col].describe())
    else:
        print("\nWarning: Not enough sugar content data available for modeling")
else:
    print("\nNo sugar content columns found")
    target_col = None

# Step 5: Prepare data for modeling
print("\nStep 5: Preparing data for modeling...")
if target_col:
    # Drop rows with missing target values
    model_df = baby_foods_processed.dropna(subset=[target_col]).copy()
    print(f"Dataset after dropping missing target values: {len(model_df)} rows")
    
    if len(model_df) < 10:
        print("Not enough data points for modeling")
        X, y = None, None
    else:
        # Select features for modeling
        feature_cols = [
            'sweetener_count', 'sweetener_position', 'fruit_count', 
            'fruit_position', 'ingredient_count', 'is_organic', 'preservative_count'
        ]
        
        # Create feature matrix X and target vector y
        X = model_df[feature_cols].copy()
        y = model_df[target_col].copy()
        
        # Handle missing values in features
        X = X.fillna({
            'sweetener_position': -1,
            'fruit_position': -1
        })
        X = X.fillna(0)  # Fill remaining NAs with 0
        
        print(f"Final dataset for modeling: {X.shape[0]} rows, {X.shape[1]} features")
        print("\nFeature summary:")
        print(X.describe())
        
        # Step 6: Train and evaluate models
        print("\nStep 6: Training prediction models...")
        if X is not None and len(X) >= 10:
            # Split the data
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            
            # Scale the features
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Define models
            models = {
                'Linear Regression': LinearRegression(),
                'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
                'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42)
            }
            
            # Train and evaluate models
            results = {}
            
            for name, model in models.items():
                print(f"\nTraining {name}...")
                model.fit(X_train_scaled, y_train)
                
                # Make predictions
                y_pred = model.predict(X_test_scaled)
                
                # Calculate metrics
                mae = mean_absolute_error(y_test, y_pred)
                rmse = np.sqrt(mean_squared_error(y_test, y_pred))
                r2 = r2_score(y_test, y_pred)
                
                print(f"{name} performance:")
                print(f"  MAE: {mae:.2f}")
                print(f"  RMSE: {rmse:.2f}")
                print(f"  R²: {r2:.2f}")
                
                # Store results
                results[name] = {
                    'model': model,
                    'y_test': y_test,
                    'y_pred': y_pred,
                    'mae': mae,
                    'rmse': rmse,
                    'r2': r2
                }
                
                # Feature importance for tree-based models
                if hasattr(model, 'feature_importances_'):
                    feature_importance = pd.DataFrame({
                        'Feature': X.columns,
                        'Importance': model.feature_importances_
                    }).sort_values('Importance', ascending=False)
                    
                    print(f"\n{name} Feature Importance:")
                    print(feature_importance)
                    
                    # Store feature importance
                    results[name]['feature_importance'] = feature_importance
            
            # Step 7: Analyze and interpret results
            print("\nStep 7: Analyzing and interpreting results...")
            # Compare model performances
            model_comparison = pd.DataFrame({
                'Model': list(results.keys()),
                'MAE': [results[model]['mae'] for model in results],
                'RMSE': [results[model]['rmse'] for model in results],
                'R²': [results[model]['r2'] for model in results]
            })
            
            print("Model Performance Comparison:")
            print(model_comparison)
            
            # Find the best performing model based on R²
            best_model_name = model_comparison.loc[model_comparison['R²'].idxmax(), 'Model']
            print(f"\nBest performing model: {best_model_name}")
            
            # Visualize feature importance for the best model
            if 'feature_importance' in results[best_model_name]:
                feature_importance = results[best_model_name]['feature_importance']
                
                plt.figure(figsize=(10, 6))
                sns.barplot(x='Importance', y='Feature', data=feature_importance)
                plt.title(f'Feature Importance for {best_model_name}')
                plt.tight_layout()
                plt.show()
                
                print("\nMost important features for predicting sugar content:")
                for i, row in feature_importance.head(3).iterrows():
                    print(f"  - {row['Feature']}: {row['Importance']:.4f}")
            
            # Draw conclusions and provide insights
            print("\nKey Insights and Recommendations:")
            
            # Most important feature
            if 'feature_importance' in results[best_model_name]:
                top_feature = feature_importance.iloc[0]['Feature']
                print(f"  - {top_feature} is the most important predictor of sugar content in baby foods")
            
            # Analyze average sugar by sweetener count
            sugar_by_sweetener = baby_foods_processed.groupby('sweetener_count')[target_col].mean()
            print("\nAverage sugar content by sweetener count:")
            for count, avg in sugar_by_sweetener.items():
                if not pd.isna(avg):
                    print(f"  - {count} sweeteners: {avg:.2f} g")
            
            # Provide recommendations
            print("\nRecommendations for parents:")
            print("  - Pay attention to the number of sweeteners in baby food ingredients")
            print("  - Check the position of sweeteners in the ingredient list - earlier means higher proportion")
            print("  - Consider that foods with fruits as early ingredients may have naturally higher sugar content")
            print("  - Choose products with simpler ingredient lists when possible")
            if 'is_organic' in model_df.columns:
                organic_sugar = model_df[model_df['is_organic'] == 1][target_col].mean()
                non_organic_sugar = model_df[model_df['is_organic'] == 0][target_col].mean()
                if not pd.isna(organic_sugar) and not pd.isna(non_organic_sugar):
                    print(f"  - {'Organic' if organic_sugar < non_organic_sugar else 'Non-organic'} baby foods tend to have lower sugar content on average")
            
            # Step 8: Create a simple function to predict sugar content for new products
            print("\nStep 8: Creating a function to predict sugar content for new products...")
            
            # Save the best model
            best_model = results[best_model_name]['model']
            
            def predict_sugar_content(ingredients, is_organic=0):
                """
                Predict sugar content for a new baby food product based on ingredients
                
                Args:
                    ingredients: String containing ingredient list
                    is_organic: 1 if product is organic, 0 otherwise
                    
                Returns:
                    Predicted sugar content
                """
                # Extract features
                sweetener_count = count_ingredient(ingredients, sweeteners)
                sweetener_pos = first_position(ingredients, sweeteners)
                fruit_count = count_ingredient(ingredients, fruits)
                fruit_pos = first_position(ingredients, fruits)
                ingredient_count = len(str(ingredients).split(','))
                preservative_count = count_ingredient(ingredients, preservatives)
                
                # Create feature vector
                features = np.array([
                    sweetener_count, 
                    sweetener_pos if sweetener_pos >= 0 else -1, 
                    fruit_count, 
                    fruit_pos if fruit_pos >= 0 else -1, 
                    ingredient_count, 
                    is_organic, 
                    preservative_count
                ]).reshape(1, -1)
                
                # Scale features
                features_scaled = scaler.transform(features)
                
                # Make prediction
                predicted_sugar = best_model.predict(features_scaled)[0]
                
                return predicted_sugar
            
            # Example usage
            print("\nExample predictions for hypothetical products:")
            
            example1 = "Organic apple puree, pear juice concentrate, cinnamon"
            predicted1 = predict_sugar_content(example1, is_organic=1)
            print(f"1. {example1}")
            print(f"   Predicted sugar content: {predicted1:.1f} g")
            
            example2 = "Water, carrots, potatoes, chicken, rice"
            predicted2 = predict_sugar_content(example2)
            print(f"2. {example2}")
            print(f"   Predicted sugar content: {predicted2:.1f} g")
            
            example3 = "Apple juice, sugar, corn syrup, flour, vanilla flavor"
            predicted3 = predict_sugar_content(example3)
            print(f"3. {example3}")
            print(f"   Predicted sugar content: {predicted3:.1f} g")
            
            print("\nProject complete! The model can now be used to predict sugar content in baby foods based on ingredients.")
        else:
            print("Not enough data for modeling. Please collect more baby food data with sugar content.")
    
else:
    print("No suitable target column found. Unable to proceed with modeling.")

print("\nEnd of analysis.")